# 🥩 고단백질 섭취 × 근력운동 × 신장기능 분석
**데이터**: CDC NHANES 2013–2014 (Kaggle)

---
## ⚙️ 사전 준비 (최초 1회)
1. [kaggle.com](https://kaggle.com) → Settings → API → **Create New Token** → `kaggle.json` 다운로드
2. **아래 첫 번째 셀만 실행** → kaggle.json 업로드 → 이후 `런타임 → 모두 실행`

> 총 소요 시간: 약 **3~5분** (데이터 다운로드 포함)

## STEP 0. Kaggle 인증

In [ ]:
from google.colab import files
uploaded = files.upload()  # kaggle.json 선택
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle 인증 완료')

## STEP 1. 환경 설정 + 데이터 다운로드

In [ ]:
# ── 라이브러리 ──────────────────────────────────────────
!pip install -q kaggle
import subprocess
subprocess.run(['apt-get','-qq','install','fonts-nanum'], check=False)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

# 한글 폰트
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 130,
                     'axes.spines.top': False, 'axes.spines.right': False})

# 컬러 팔레트
PALETTE = ['#2D6A4F','#52B788','#95D5B2','#D8F3DC',
           '#E76F51','#F4A261','#264653','#457B9D']
ACCENT  = '#E76F51'
BLUE    = '#457B9D'
GREEN   = '#2D6A4F'

# ── Kaggle 데이터 다운로드 ──────────────────────────────
!kaggle datasets download -d cdc/national-health-and-nutrition-examination-survey --unzip -q

import glob
csvs = sorted(glob.glob('*.csv'))
print('다운로드된 파일:', csvs)
print('✅ 환경 준비 완료')

## STEP 2. 데이터 로드 & 병합

In [ ]:
# NHANES 파일 탐색 (파일명이 데이터셋마다 다를 수 있음)
def find_file(keywords):
    for kw in keywords:
        for f in csvs:
            if kw.lower() in f.lower():
                return f
    return None

# 각 파일 찾기
demo_f  = find_file(['demo'])
diet_f  = find_file(['dr1tot','diet','nutrient'])
lab_f   = find_file(['biopro','lab','biochem'])
paq_f   = find_file(['paq','physical','activity'])
bmx_f   = find_file(['bmx','body'])

print('인구통계:', demo_f)
print('식이섭취:', diet_f)
print('검사수치:', lab_f)
print('신체활동:', paq_f)
print('신체계측:', bmx_f)

In [ ]:
# ── 파일 로드 ────────────────────────────────────────────
dfs = {}
for name, path in [('demo',demo_f),('diet',diet_f),('lab',lab_f),
                    ('paq',paq_f),('bmx',bmx_f)]:
    if path:
        dfs[name] = pd.read_csv(path)
        print(f'{name}: {dfs[name].shape}  컬럼 예시: {list(dfs[name].columns[:5])}')
    else:
        print(f'⚠ {name} 파일 없음')

In [ ]:
# ── SEQN 기준 병합 ──────────────────────────────────────
from functools import reduce

frames = [df for df in dfs.values() if df is not None]
# 공통 식별자 찾기 (SEQN 또는 첫 번째 컬럼)
id_col = 'SEQN' if 'SEQN' in frames[0].columns else frames[0].columns[0]
print(f'병합 기준 컬럼: {id_col}')

raw = reduce(lambda l, r: pd.merge(l, r, on=id_col, how='inner'), frames)
print(f'\n병합 결과: {raw.shape[0]:,}명 × {raw.shape[1]}개 컬럼')
raw.head(3)

## STEP 3. 핵심 변수 추출 & 전처리

In [ ]:
# ── 변수 자동 탐색 ──────────────────────────────────────
cols = raw.columns.str.upper()

def pick(keywords, cols=cols, raw=raw):
    """키워드 리스트로 컬럼 탐색"""
    for kw in keywords:
        matches = [c for c in cols if kw in c]
        if matches: return matches[0]
    return None

VAR = {
    # 독립변수: 단백질
    'protein_total'   : pick(['DR1TPROT','DRXTPROT','PROTEIN']),
    'protein_animal'  : pick(['DR1TACAR','ANIMAL']),
    'protein_plant'   : pick(['DR1TPCAR','PLANT']),
    'energy'          : pick(['DR1TKCAL','ENERGY','KCAL']),
    # 독립변수: 운동
    'strength_days'   : pick(['PAD660','PAQ650','STRENGTH','MUSCLE']),
    'cardio_min'      : pick(['PAD615','PAQ610','CARDIO','AEROBIC']),
    # 종속변수: 신장기능
    'BUN'             : pick(['LBXBUN','BUN','UREA']),
    'creatinine'      : pick(['LBXSCR','CREATININE','SCR']),
    # 보정변수
    'BMI'             : pick(['BMXBMI','BMI']),
    'age'             : pick(['RIDAGEYR','AGE']),
    'sex'             : pick(['RIAGENDR','GENDER','SEX']),
    'weight_kg'       : pick(['WTMEC2YR','WTINT2YR','WEIGHT']),
}

print('변수 매핑 결과:')
for k, v in VAR.items():
    status = '✔' if v else '✗ 없음'
    print(f'  {k:20s} → {v or "없음"} {status}')

In [ ]:
# ── 분석 데이터프레임 구성 ──────────────────────────────
use = {k: v for k, v in VAR.items() if v}
df = raw[[v for v in use.values()]].copy()
df.columns = list(use.keys())

# 성인(19세+) 필터
if 'age' in df.columns:
    df = df[df['age'] >= 19]

print(f'성인 필터 후: {len(df):,}명')

# ── 전처리 ──────────────────────────────────────────────
# 1. 결측치 5% 이상 컬럼 → 컬럼 제거, 미만 → 행 제거
miss_rate = df.isnull().mean()
drop_cols = miss_rate[miss_rate >= 0.5].index.tolist()
if drop_cols:
    print(f'결측 50% 초과 컬럼 제거: {drop_cols}')
    df = df.drop(columns=drop_cols)

before = len(df)
core_cols = [c for c in ['protein_total','BUN','creatinine'] if c in df.columns]
df = df.dropna(subset=core_cols)
print(f'핵심변수 결측 제거: {before - len(df)}명 제거 → {len(df):,}명 남음')

# 2. IQR 이상치 처리 (Winsorize: 상한/하한 값으로 대체)
def winsorize(series, q=0.01):
    lo, hi = series.quantile(q), series.quantile(1-q)
    return series.clip(lo, hi)

for col in ['protein_total','BUN','creatinine','BMI']:
    if col in df.columns:
        df[col] = winsorize(df[col])

# 3. 파생변수: 체중 보정 단백질 섭취량 (g/kg BMI)
if 'protein_total' in df.columns and 'energy' in df.columns:
    df['protein_pct_energy'] = (df['protein_total'] * 4 / df['energy'].replace(0, np.nan)) * 100

# 4. 신기능 저하 분류 (크레아티닌 기준: 남성 >1.3, 여성 >1.1 mg/dL)
if 'creatinine' in df.columns and 'sex' in df.columns:
    # NHANES: sex=1 남성, 2 여성
    df['kidney_risk'] = np.where(
        ((df['sex']==1) & (df['creatinine'] > 1.3)) |
        ((df['sex']==2) & (df['creatinine'] > 1.1)),
        '고위험', '정상'
    )
elif 'creatinine' in df.columns:
    df['kidney_risk'] = np.where(df['creatinine'] > 1.2, '고위험', '정상')

# 5. 단백질 섭취 그룹 (3분위)
if 'protein_total' in df.columns:
    df['protein_group'] = pd.qcut(df['protein_total'], 3,
                                   labels=['저단백','중단백','고단백'])

print(f'\n✅ 전처리 완료: {len(df):,}명')
print(df[core_cols + ['kidney_risk']].describe().round(2))
if 'kidney_risk' in df.columns:
    print('\n신기능 분류:', df['kidney_risk'].value_counts().to_dict())

## STEP 4. 시각화 — 그래프 모음

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 1: 단백질 섭취량 분포 + 신기능 그룹 비교
# ════════════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 3, wspace=0.35)

# 1-1 히스토그램
ax1 = fig.add_subplot(gs[0])
if 'protein_total' in df.columns:
    n, bins, patches = ax1.hist(df['protein_total'].dropna(), bins=35,
                                 color=GREEN, edgecolor='white', linewidth=0.5, alpha=0.85)
    ax1.axvline(df['protein_total'].mean(), color=ACCENT, lw=2, ls='--',
                label=f'평균 {df["protein_total"].mean():.0f}g')
    ax1.axvline(df['protein_total'].median(), color=BLUE, lw=2, ls=':',
                label=f'중앙값 {df["protein_total"].median():.0f}g')
    ax1.set_xlabel('1일 총 단백질 섭취량 (g)', fontsize=11)
    ax1.set_ylabel('인원 수', fontsize=11)
    ax1.set_title('단백질 섭취량 분포', fontsize=13, fontweight='bold', pad=12)
    ax1.legend(fontsize=9)

# 1-2 BUN 박스플롯 (단백질 그룹별)
ax2 = fig.add_subplot(gs[1])
if 'protein_group' in df.columns and 'BUN' in df.columns:
    data_bp = [df[df['protein_group']==g]['BUN'].dropna()
               for g in ['저단백','중단백','고단백']]
    bp = ax2.boxplot(data_bp, patch_artist=True, notch=True,
                     medianprops=dict(color='white', lw=2.5),
                     flierprops=dict(marker='o', markersize=3, alpha=0.3, color=ACCENT))
    box_colors = ['#95D5B2','#52B788','#2D6A4F']
    for patch, c in zip(bp['boxes'], box_colors):
        patch.set_facecolor(c); patch.set_alpha(0.85)
    ax2.set_xticklabels(['저단백','중단백','고단백'], fontsize=10)
    ax2.set_ylabel('혈중 요소질소 BUN (mg/dL)', fontsize=11)
    ax2.set_title('단백질 그룹별 BUN 수치', fontsize=13, fontweight='bold', pad=12)

# 1-3 크레아티닌 박스플롯 (단백질 그룹별)
ax3 = fig.add_subplot(gs[2])
if 'protein_group' in df.columns and 'creatinine' in df.columns:
    data_cr = [df[df['protein_group']==g]['creatinine'].dropna()
               for g in ['저단백','중단백','고단백']]
    bp2 = ax3.boxplot(data_cr, patch_artist=True, notch=True,
                      medianprops=dict(color='white', lw=2.5),
                      flierprops=dict(marker='o', markersize=3, alpha=0.3, color=ACCENT))
    for patch, c in zip(bp2['boxes'], ['#F4A261','#E76F51','#C45230']):
        patch.set_facecolor(c); patch.set_alpha(0.85)
    ax3.set_xticklabels(['저단백','중단백','고단백'], fontsize=10)
    ax3.set_ylabel('혈청 크레아티닌 (mg/dL)', fontsize=11)
    ax3.set_title('단백질 그룹별 크레아티닌', fontsize=13, fontweight='bold', pad=12)

fig.suptitle('단백질 섭취량 & 신장 기능 지표', fontsize=15, fontweight='bold', y=1.02)
plt.savefig('plot1_protein_kidney.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 1 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 2: 산점도 + 회귀선 (단백질 vs 신장지표)
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('단백질 섭취량 vs 신장 기능 지표 산점도', fontsize=14, fontweight='bold', y=1.02)

for ax, ycol, ylabel, color in zip(
    axes,
    ['BUN', 'creatinine'],
    ['BUN (mg/dL)', '크레아티닌 (mg/dL)'],
    [GREEN, ACCENT]
):
    if 'protein_total' not in df.columns or ycol not in df.columns:
        continue
    sub = df[['protein_total', ycol, 'kidney_risk']].dropna()
    x, y = sub['protein_total'], sub[ycol]

    # 신기능 그룹별 색상
    colors_pt = sub['kidney_risk'].map({'정상': color, '고위험': ACCENT}) if 'kidney_risk' in sub else color
    ax.scatter(x, y, c=colors_pt, s=12, alpha=0.35, edgecolors='none')

    # 회귀선
    sl, ic, r, p, _ = stats.linregress(x, y)
    xr = np.linspace(x.min(), x.max(), 200)
    ax.plot(xr, sl*xr+ic, color='#264653', lw=2.2,
            label=f'r = {r:.3f}  (p {"< 0.001" if p<0.001 else f"= {p:.3f}"})')
    ax.set_xlabel('1일 총 단백질 섭취량 (g)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f'단백질 vs {ylabel}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

    # 범례 패치
    p1 = mpatches.Patch(color=color, alpha=0.6, label='정상')
    p2 = mpatches.Patch(color=ACCENT, alpha=0.6, label='고위험')
    ax.legend(handles=[p1, p2, plt.Line2D([0],[0], color='#264653', lw=2,
              label=f'r={r:.3f}, p={"<0.001" if p<0.001 else f"{p:.3f}"}')], fontsize=8)

plt.tight_layout()
plt.savefig('plot2_scatter_regression.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 2 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 3: 상관관계 히트맵 (아름다운 버전)
# ════════════════════════════════════════════════════════
num_cols = [c for c in ['protein_total','protein_pct_energy',
                         'BUN','creatinine','BMI',
                         'strength_days','cardio_min','age']
            if c in df.columns]

col_labels = {
    'protein_total':'단백질\n섭취(g)', 'protein_pct_energy':'단백질\n에너지(%)',
    'BUN':'BUN', 'creatinine':'크레아티닌',
    'BMI':'BMI', 'strength_days':'근력운동\n일수',
    'cardio_min':'유산소\n시간(분)', 'age':'연령'
}

corr = df[num_cols].corr()
labels = [col_labels.get(c, c) for c in num_cols]
corr.index = labels; corr.columns = labels

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = LinearSegmentedColormap.from_list('custom',
       ['#E76F51','#F4F1DE','#2D6A4F'], N=256)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            center=0, vmin=-1, vmax=1,
            linewidths=0.8, linecolor='white',
            square=True, annot_kws={'size':10, 'weight':'bold'}, ax=ax,
            cbar_kws={'shrink':0.8, 'label':'상관계수 r'})
ax.set_title('변수 간 상관관계 히트맵', fontsize=14, fontweight='bold', pad=16)
ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.savefig('plot3_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ 그래프 3 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 4: 신기능 고위험군 vs 정상 — 바이올린 플롯
# ════════════════════════════════════════════════════════
if 'kidney_risk' in df.columns:
    compare_cols = [c for c in ['protein_total','BMI','age',
                                 'strength_days','cardio_min'] if c in df.columns]
    col_names = ['단백질섭취(g)','BMI','연령','근력운동일수','유산소시간(분)']

    fig, axes = plt.subplots(1, len(compare_cols), figsize=(4.5*len(compare_cols), 6))
    if len(compare_cols) == 1: axes = [axes]
    fig.suptitle('신기능 고위험군 vs 정상군 비교', fontsize=14, fontweight='bold', y=1.02)

    for ax, col, name in zip(axes, compare_cols, col_names):
        sub = df[['kidney_risk', col]].dropna()
        groups = [sub[sub['kidney_risk']==g][col].values for g in ['정상','고위험']]

        parts = ax.violinplot(groups, positions=[1,2], showmedians=True,
                               showextrema=False)
        colors_v = [GREEN, ACCENT]
        for pc, c in zip(parts['bodies'], colors_v):
            pc.set_facecolor(c); pc.set_alpha(0.75); pc.set_edgecolor('white')
        parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2)

        # t-검정
        t, p = stats.ttest_ind(*groups)
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        ymax = max([g.max() for g in groups if len(g)])
        ax.plot([1,2],[ymax*1.05]*2, color='#264653', lw=1.2)
        ax.text(1.5, ymax*1.08, sig, ha='center', fontsize=12, color='#264653', fontweight='bold')

        ax.set_xticks([1,2]); ax.set_xticklabels(['정상','고위험'], fontsize=10)
        ax.set_ylabel(name, fontsize=10)
        ax.set_title(name, fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('plot4_violin_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ 그래프 4 저장  (* p<0.05, ** p<0.01, *** p<0.001, ns=비유의)')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 5: BMI × 단백질 × 신기능 — 버블 차트
# ════════════════════════════════════════════════════════
if all(c in df.columns for c in ['BMI','protein_total','creatinine','kidney_risk']):
    fig, ax = plt.subplots(figsize=(10, 7))

    for grp, color, alpha in [('정상', GREEN, 0.4), ('고위험', ACCENT, 0.7)]:
        sub = df[df['kidney_risk']==grp].dropna(subset=['BMI','protein_total','creatinine'])
        sub = sub.sample(min(800, len(sub)), random_state=42)
        ax.scatter(sub['protein_total'], sub['BMI'],
                   s=sub['creatinine']*60, c=color, alpha=alpha,
                   edgecolors='white', linewidth=0.4, label=f'{grp} (버블=크레아티닌)')

    ax.set_xlabel('1일 단백질 섭취량 (g)', fontsize=12)
    ax.set_ylabel('체질량지수 BMI (kg/m²)', fontsize=12)
    ax.set_title('단백질 섭취 × BMI × 신기능 위험도\n(버블 크기 = 크레아티닌 수치)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, markerscale=0.7)
    ax.axhline(25, color='#264653', lw=1.2, ls='--', alpha=0.5, label='BMI 25 기준선')
    ax.axhline(30, color=ACCENT,    lw=1.2, ls='--', alpha=0.5, label='BMI 30 기준선')

    plt.tight_layout()
    plt.savefig('plot5_bubble_chart.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ 그래프 5 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 6: 운동 × 단백질 2×2 히트맵 (신기능 위험 비율)
# ════════════════════════════════════════════════════════
if all(c in df.columns for c in ['protein_group','kidney_risk']) and \
   any(c in df.columns for c in ['strength_days','cardio_min']):

    ex_col = 'strength_days' if 'strength_days' in df.columns else 'cardio_min'
    ex_label = '근력운동 일수' if ex_col == 'strength_days' else '유산소 운동 시간(분)'

    df_sub = df[['protein_group', ex_col, 'kidney_risk']].dropna()
    df_sub['ex_group'] = pd.qcut(df_sub[ex_col], 2,
                                  labels=['운동 적음','운동 많음'],
                                  duplicates='drop')

    pivot = df_sub.groupby(['protein_group','ex_group'])['kidney_risk'].apply(
        lambda x: (x=='고위험').mean() * 100
    ).unstack()

    fig, ax = plt.subplots(figsize=(7, 5))
    cmap2 = LinearSegmentedColormap.from_list('risk', ['#D8F3DC','#E76F51'], N=256)
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap=cmap2,
                linewidths=1.5, linecolor='white',
                annot_kws={'size':14, 'weight':'bold'},
                cbar_kws={'label':'신기능 고위험 비율 (%)'}, ax=ax)
    ax.set_xlabel(ex_label, fontsize=11)
    ax.set_ylabel('단백질 섭취 그룹', fontsize=11)
    ax.set_title(f'단백질 섭취 × {ex_label}별\n신기능 고위험 비율 (%)',
                 fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('plot6_risk_heatmap.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ 그래프 6 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 7: 연령대별 BUN 변화 + 단백질 섭취 라인 차트
# ════════════════════════════════════════════════════════
if all(c in df.columns for c in ['age','BUN','protein_total']):
    df['age_group'] = pd.cut(df['age'], bins=[18,29,39,49,59,69,120],
                              labels=['20대','30대','40대','50대','60대','70대+'])
    age_stat = df.groupby('age_group', observed=True).agg(
        BUN_mean=('BUN','mean'),
        BUN_se=('BUN', lambda x: x.std()/np.sqrt(len(x))),
        protein_mean=('protein_total','mean')
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax2 = ax1.twinx()

    x = range(len(age_stat))
    ax1.bar(x, age_stat['BUN_mean'], color=GREEN, alpha=0.75, width=0.4, label='BUN 평균')
    ax1.errorbar(x, age_stat['BUN_mean'], yerr=age_stat['BUN_se']*1.96,
                 fmt='none', color='#264653', capsize=4, lw=1.5)
    ax2.plot(x, age_stat['protein_mean'], color=ACCENT, lw=2.5,
             marker='o', markersize=8, label='단백질 평균(g)')

    ax1.set_xticks(x); ax1.set_xticklabels(age_stat['age_group'], fontsize=11)
    ax1.set_ylabel('BUN 평균 (mg/dL)', fontsize=11, color=GREEN)
    ax2.set_ylabel('단백질 섭취량 평균 (g)', fontsize=11, color=ACCENT)
    ax1.set_title('연령대별 BUN 수치 & 단백질 섭취량', fontsize=13, fontweight='bold')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

    plt.tight_layout()
    plt.savefig('plot7_age_trend.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ 그래프 7 저장')

In [ ]:
# ════════════════════════════════════════════════════════
# 그래프 8: 신기능 고위험 비율 — 단백질 구간별 스택 막대
# ════════════════════════════════════════════════════════
if 'protein_group' in df.columns and 'kidney_risk' in df.columns:
    risk_pct = df.groupby('protein_group', observed=True)['kidney_risk'].value_counts(
        normalize=True).unstack().fillna(0) * 100

    fig, ax = plt.subplots(figsize=(8, 5))
    bottom = np.zeros(len(risk_pct))
    colors_s = {'정상': '#95D5B2', '고위험': ACCENT}

    for grp, color in colors_s.items():
        if grp in risk_pct.columns:
            vals = risk_pct[grp].values
            bars = ax.bar(risk_pct.index, vals, bottom=bottom,
                          color=color, edgecolor='white', linewidth=0.8, label=grp)
            for bar, val in zip(bars, vals):
                if val > 5:
                    ax.text(bar.get_x()+bar.get_width()/2,
                            bar.get_y()+bar.get_height()/2,
                            f'{val:.1f}%', ha='center', va='center',
                            fontsize=12, fontweight='bold', color='white')
            bottom += vals

    ax.set_xlabel('단백질 섭취 그룹', fontsize=12)
    ax.set_ylabel('비율 (%)', fontsize=12)
    ax.set_title('단백질 섭취 그룹별 신기능 위험 비율', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylim(0, 105)

    plt.tight_layout()
    plt.savefig('plot8_stacked_bar.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ 그래프 8 저장')

## STEP 5. 통계 검정 요약

In [ ]:
print('='*60)
print('📊 통계 검정 결과 요약')
print('='*60)

# 1. 단백질 그룹별 BUN/크레아티닌 차이 (ANOVA)
for yc, yl in [('BUN','BUN'), ('creatinine','크레아티닌')]:
    if 'protein_group' in df.columns and yc in df.columns:
        groups = [df[df['protein_group']==g][yc].dropna()
                  for g in ['저단백','중단백','고단백']]
        f, p = stats.f_oneway(*groups)
        print(f'\n단백질 그룹 × {yl} ANOVA')
        print(f'  F = {f:.3f}, p = {p:.4f}  {"*** 유의" if p<0.001 else "** 유의" if p<0.01 else "* 유의" if p<0.05 else "비유의"}')
        for g, grp in zip(['저단백','중단백','고단백'], groups):
            print(f'  {g}: 평균={grp.mean():.2f} ± {grp.std():.2f}')

# 2. 피어슨 상관
print('\n\n단백질 섭취량 vs 신장기능 상관계수')
for yc, yl in [('BUN','BUN'), ('creatinine','크레아티닌')]:
    if 'protein_total' in df.columns and yc in df.columns:
        r, p = stats.pearsonr(df['protein_total'].dropna(),
                               df[yc].dropna()[:len(df['protein_total'].dropna())])
        print(f'  {yl}: r = {r:.4f}, p = {p:.4f}')

# 3. 카이제곱 (단백질 그룹 × 신기능)
if 'protein_group' in df.columns and 'kidney_risk' in df.columns:
    ct = pd.crosstab(df['protein_group'], df['kidney_risk'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f'\n\n단백질 그룹 × 신기능 카이제곱 검정')
    print(f'  chi2 = {chi2:.3f}, df = {dof}, p = {p:.4f}')
    print(ct)

print('\n✅ 통계 검정 완료')

## STEP 6. 결과 저장

In [ ]:
df.to_csv('분석데이터_최종.csv', index=False, encoding='utf-8-sig')
desc = df.select_dtypes(include='number').describe().round(3)
desc.to_csv('기술통계_요약.csv', encoding='utf-8-sig')

saved = ['분석데이터_최종.csv','기술통계_요약.csv'] + \
        [f'plot{i}_{n}.png' for i,n in enumerate(
         ['protein_kidney','scatter_regression','heatmap',
          'violin_comparison','bubble_chart',
          'risk_heatmap','age_trend','stacked_bar'], 1)]

print('\n📁 저장 파일 목록:')
for f in saved:
    if os.path.exists(f):
        kb = os.path.getsize(f)/1024
        print(f'  ✔ {f}  ({kb:.0f} KB)')

print('\n✅ 분석 완료!')
print('\n📌 다음 단계: 로지스틱 회귀(OR 95%CI) 또는 다중회귀분석으로 확장 가능')

---
## 📋 그래프 목록
| 파일 | 내용 |
|------|------|
| plot1_protein_kidney.png | 단백질 분포 히스토그램 + 그룹별 BUN/크레아티닌 박스플롯 |
| plot2_scatter_regression.png | 단백질 vs BUN/크레아티닌 산점도 + 회귀선 |
| plot3_heatmap.png | 변수 간 상관관계 히트맵 (커스텀 컬러) |
| plot4_violin_comparison.png | 신기능 정상 vs 고위험 바이올린 플롯 + 유의성 표시 |
| plot5_bubble_chart.png | 단백질 × BMI × 크레아티닌 버블 차트 |
| plot6_risk_heatmap.png | 단백질×운동 그룹별 신기능 고위험 비율 히트맵 |
| plot7_age_trend.png | 연령대별 BUN & 단백질 섭취량 이중축 차트 |
| plot8_stacked_bar.png | 단백질 그룹별 신기능 위험 비율 누적 막대 |